## Setup

In [1]:
import numpy as np
import pandas as pd
from src.kmeans_parallel import kmeans_parallel
from src.launch_cluster import launch_cluster, shutdown_cluster
from src.data_loader import load_dataset
from src.benchmark import run_single_test, run_benchmark, calculate_inertia, combinations_fn

import time

In [2]:
# --- Cluster ---
N_WORKERS = 8      # tra 1 e 8 (nodi disponibili in launch_cluster.py)
NUM_PARTITIONS = 8 * N_WORKERS   # regola empirica: >= n_threads_per_worker * n_workers
# --- Algoritmo k-means|| ---
#K = 500                # numero di cluster finali
#L = 250                # oversampling factor (assoluto). In alternativa: L = round(L_OVER_K * K)
#R = 10                  # numero di round dell'inizializzazione parallela
#MAX_ITER_FIT = 100      # iterazioni massime della fase di Lloyd's (fit)

SEED = 42

In [3]:
# --- Dataset ---
DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
# link used by sklearn function fetch_kddcup99 (original link gives 403 error)
#***for 10% dataset***

DATASET_URL_FULL="https://ndownloader.figshare.com/files/5976045"
#***FULL DATASET***

RAW_GZ_PATH_10PC   = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz" # compressed (.gz) dataset file
PARQUET_PATH_10PC = '/tmp/kddcup_data.parquet' # single Parquet file (ie a compressed format file but partition-able, unlike .gz) on master

RAW_GZ_PATH_FULL = "/home/ubuntu/backup/libero_development/data/kddcup_data_full.gz"
PARQUET_PATH_FULL = '/tmp/kddcup_data_full.parquet'

# column names of KDD dataset, from source code of the above sklearn function;
# "protocol_type","service","flag" are non-numeric so they will be dropped later,
# as will be "label" and the constant column 'num_outbound_cmds' 
# (10% dataset might have more constant columns such as 'is_host_login')
COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label"
]

#### IF ALREADY EXISTING CLUSTER:

In [4]:
# DO NOT RUN if already existing!
cluster, client = launch_cluster(N_WORKERS)

Inizializzazione del cluster SSH con 8 worker...
Worker selezionati: ['10.67.22.254', '10.67.22.34', '10.67.22.145', '10.67.22.121', '10.67.22.192', '10.67.22.18', '10.67.22.187', '10.67.22.48']


2026-08-31 18:44:50,180 - distributed.deploy.ssh - INFO - 2026-08-31 18:44:50,179 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-08-31 18:44:50,208 - distributed.deploy.ssh - INFO - 2026-08-31 18:44:50,207 - distributed.scheduler - INFO - State start
2026-08-31 18:44:50,212 - distributed.deploy.ssh - INFO - 2026-08-31 18:44:50,211 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.67.22.194:8786
2026-08-31 18:44:52,154 - distributed.deploy.ssh - INFO - 2026-08-31 18:44:52,151 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.254:46555'
2026-08-31 18:44:52,156 - distributed.deploy.ssh - INFO - 2026-08-31 18:44:52,158 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.18:40935'
2026-08-31 18:44:52,160 - distributed.deploy.ssh - INFO - 2026-08-31 18:44:52,156 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.121

Cluster avviato e connessione stabilita con successo!



In [4]:
from dask.distributed import Client

SCHEDULER_ADDRESS = "tcp://10.67.22.194:8786"

try:
    client = Client(SCHEDULER_ADDRESS, timeout="10s")
    print("Connected to cluster successfully!")
    print(f"Dask Dashboard link: {client.dashboard_link}")

except Exception as e:
    print(f"Connection error: {e}")

Connection error: Timed out trying to connect to tcp://10.67.22.194:8786 after 10 s


## Load dataset (10%)

In [23]:
from src.data_loader import load_dataset

DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
RAW = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz"   # già in cache, niente download
PQ  = "/tmp/kddcup_data.parquet"
COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label"
]

In [24]:
#10 percent:

start=time.time()
X_bag_10_percent, (mean_ar, std_ar)=load_dataset(n_partitions=NUM_PARTITIONS,
                   client=client,
                   dataset_url=DATASET_URL_10PC,
                   raw_gz_path=RAW_GZ_PATH_10PC,
                   parquet_path=PARQUET_PATH_10PC,
                   parquet_path_workers=PARQUET_PATH_10PC,
                   col_names=COL_NAMES,
                   force_download=False)
end=time.time()
elapsed=end-start
print(f"Time elapsed: {elapsed:.2f} s")

Converting .gz -> Parquet chunk sizes...
Parquet file created (compressed with snappy).
Number of partitions before preprocessing: 64
Constant columns: ['num_outbound_cmds', 'is_host_login']
Computing global mean and std (first pass over data)...
Distributed bag created with 32 partitions.
Number of samples: 494021


In [5]:
# full:

start=time.time()
X_bag_full, (mean_ar, std_ar)=load_dataset(n_partitions=NUM_PARTITIONS,
                   client=client,
                   dataset_url=DATASET_URL_FULL,#DATASET_URL_10PC,
                   raw_gz_path=RAW_GZ_PATH_FULL,#RAW_GZ_PATH,
                   parquet_path=PARQUET_PATH_FULL,#PARQUET_PATH,
                   parquet_path_workers=PARQUET_PATH_FULL,#PARQUET_PATH,
                   col_names=COL_NAMES,
                   force_download=True)
end=time.time()
elapsed=end-start
print(f"Time elapsed: {elapsed:.2f} s")

Converting .gz -> Parquet chunk sizes...
Parquet file created (compressed with snappy).
Number of partitions before preprocessing: 64
Constant columns: ['num_outbound_cmds']
Computing global mean and std (first pass over data)...
Distributed bag created with 32 partitions.
Number of samples: 4898431
Time elapsed: 69.03 s


## Run experiments

#### Varying number of L/K

In [9]:
combos = [                      
    #(N_WORKERS, 32, l_over_k, R),   # under-partitioned
    #(N_WORKERS, 64, l_over_k, R),   # balanced (1 part/thread)
    #(N_WORKERS, 128, 1, 5),   # fatto
    #(N_WORKERS, 128, 2, 5),  #  fatto 5 run
    (N_WORKERS, 256, 10, 5),
    #(N_WORKERS, 128, 0.5, 5), # fatto
    #(N_WORKERS, 128, 0.1, 5), # fatto
]
K_VALUES=[500]
#K_VALUES=[500, 1000]

MAX_ITER_FIT=80 # for best convergence (conv criterion??)

avg_iters=5

In [10]:
dataset_bag = X_bag_full
df = run_benchmark(client, X_bag=dataset_bag, combinations=combos,
                   k_values=K_VALUES, label="num_partitions_full",
                   max_iter_fit=MAX_ITER_FIT, seed=SEED, averaging_iterations=avg_iters)

Testing: k=500, workers=8, partitions=256, l=5000 (l/k=10), r=5
 Iterating 5 times.
Iteration 0


KilledWorker: Attempted to run task '_update_state-cf2fe1ca-0ad5-4559-a337-825500684689' on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://10.67.22.254:38163. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

## Different numebr of and partitions

In [11]:
combos = [                      
    (N_WORKERS, 32, 1, 5),   # under-partitioned
    (N_WORKERS, 64, 1, 5),   # balanced (1 part/thread)
    #(N_WORKERS, 65, l_over_k, R),   # imbalanced
    (N_WORKERS, 128, 1, 5),  # over-partitioned
    (N_WORKERS, 256, 1, 5),
    (N_WORKERS, 512, 1, 5),
    (N_WORKERS, 1024, 1, 5),
    (N_WORKERS, 2048, 1, 5),
]

K_VALUES=[500]
#K_VALUES=[500, 1000]

MAX_ITER_FIT=80 # for best convergence (conv criterion??)

avg_iters=3


In [ ]:
dataset_bag = X_bag_full
df = run_benchmark(client, X_bag=dataset_bag, combinations=combos,
                   k_values=K_VALUES, label="num_partitions_full",
                   max_iter_fit=MAX_ITER_FIT, seed=SEED, averaging_iterations=avg_iters)

Testing: k=500, workers=8, partitions=32, l=500 (l/k=1), r=5
 Iterating 3 times.
Iteration 0


### Tables 3 and 4

In [19]:
from src.paper_experiments import run_table34, table34_cost_table, table34_time_table

### Figure 5.1

### Figure 5.2

## Cluster shutdown

In [5]:
shutdown_cluster(cluster, client)

Cluster e client chiusi.
